# 第 1 章 · 从一个循环开始

**这一章你会得到什么**：亲手写出 Agent 的核心循环骨架，然后在真实 `default.py` 里认出同一个骨架。

本章只回答一个工程问题：**一个语言模型如何在外部世界里连续做事？** 答案能压缩成一条循环：

```text
任务 -> 模型提出动作 -> 执行动作 -> 把结果放回历史 -> 模型再提出下一步 -> ...
```

这条循环就是 Agent Runtime 的骨架。Prompt、工具、记忆、评测、沙箱，都是在这个骨架上加东西。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

建议 IDE 打开 `.py` 对照读（notebook 里点击跳转不好使）：

- `src/minisweagent/agents/default.py` **L88–122** — `run()` 主循环（本章主角）
- `src/minisweagent/agents/default.py` **L124–126** — `step()`（= query + execute_actions）
- `src/minisweagent/environments/local.py` **L45–56** — `_check_finished()` 完成信号检测
- `src/minisweagent/exceptions.py` **L1–27** — 异常家族（`Submitted` 等）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [18]:
# 环境自检：把源码目录加入 sys.path，并切到仓库根目录
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("Python:", sys.version.split()[0])
print("mini-SWE-agent:", minisweagent.__version__)
print("仓库根目录:", REPO)

Python: 3.13.14
mini-SWE-agent: 2.4.5
仓库根目录: /Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent


## 概念：三个不能混淆的角色

- **Model**：根据当前消息历史，提出下一步动作（它不会真的执行任何东西）
- **Environment**：把动作施加到真实世界，返回结果
- **Agent / Runtime**：组织循环，把 Model 和 Environment 接起来

模型只会提出类似 `{"type": "bash", "command": "ls"}` 的动作，真正执行的是 Environment。

## 动手 1：补全一个 30 行的玩具 Agent

下面是一个不依赖任何模型 SDK 的玩具循环。`FakeModel` 按剧本顺序吐回复，`FakeEnv` 假装执行命令。
**你的任务**：把 `TODO` 补完——让循环在遇到 `submit` 动作时结束。

> 完整参考答案在 `labs/01_toy_agent.py`，卡住了再看。

In [19]:
class FakeModel:
    """按剧本顺序返回回复，模拟“每轮给一个新动作”。"""
    def __init__(self, script):
        self.script = script
        self.i = -1
    def query(self, messages):
        self.i += 1
        return self.script[self.i]

class FakeEnv:
    def execute(self, action):
        # TODO: 取出 action["command"]，返回一段假的执行输出字符串
        cmd = action["command"]
        return f"执行了: {cmd}"

def toy_run(model, env, task):
    messages = [{"role": "user", "content": task}]
    while True:
        reply = model.query(messages)
        messages.append(reply)
        if reply["action"]["type"] == "submit":   # TODO 想清楚：谁负责结束循环？
            return messages
        result = env.execute(reply["action"])
        messages.append({"role": "tool", "content": result})

## 运行看结果

下面是一份可直接运行的**参考实现**。先自己写上面那格，再运行这格核对输出。

In [20]:
class FakeModelRef:
    def __init__(self, script):
        self.script, self.i = script, -1
    def query(self, messages):
        self.i += 1
        return self.script[self.i]

class FakeEnvRef:
    def execute(self, action):
        return f"[执行 {action['command']} 的输出]"

def toy_run_ref(model, env, task):
    messages = [{"role": "user", "content": task}]
    while True:
        reply = model.query(messages)
        messages.append(reply)
        if reply["action"]["type"] == "submit":
            return messages
        result = env.execute(reply["action"])
        messages.append({"role": "tool", "content": result})

script = [
    {"role": "assistant", "content": "先看看目录", "action": {"type": "bash", "command": "pwd"}},
    {"role": "assistant", "content": "列出文件", "action": {"type": "bash", "command": "ls"}},
    {"role": "assistant", "content": "任务完成", "action": {"type": "submit", "command": ""}},
]
for m in toy_run_ref(FakeModelRef(script), FakeEnvRef(), "看看目录里有什么"):
    print(m["role"], "|", m["content"])

user | 看看目录里有什么
assistant | 先看看目录
tool | [执行 pwd 的输出]
assistant | 列出文件
tool | [执行 ls 的输出]
assistant | 任务完成


## 观察点

看输出的 role 序列：`user → assistant → tool → assistant → tool → assistant`。
这就是 Agent 的灵魂。注意两件事：

1. **是模型主动发 `submit` 循环才停**——如果剧本里没有 submit，`FakeModel` 会一直被要求取下一条，最终 `IndexError`。“谁负责结束”是个真实的架构问题。
2. **`command` 之所以重要，是因为 `FakeEnv` 真的去读了它**。字段有没有用，取决于有没有代码读它。

## 连回真实源码：`DefaultAgent.run()`

真实实现里，主循环在 `run()`，不在 `step()`。运行下面这格，对照你的玩具循环。

In [21]:
# 小工具：带行号打印源码切片（相当于 nl + sed）
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

In [22]:
show_source("src/minisweagent/agents/default.py", 88, 122)

 88      def run(self, task: str = "", **kwargs) -> dict:
 89          """Run step() until agent is finished. Returns dictionary with exit_status, submission keys."""
 90          self.extra_template_vars |= {"task": task, **kwargs}
 91          self.messages = []
 92          self.add_messages(
 93              self.model.format_message(role="system", content=self._render_template(self.config.system_template)),
 94              self.model.format_message(role="user", content=self._render_template(self.config.instance_template)),
 95          )
 96          while True:
 97              try:
 98                  self.step()
 99                  self.n_consecutive_format_errors = 0  # reset on any clean step
100              except FormatError as e:
101                  self.n_consecutive_format_errors += 1
102                  if 0 < self.config.max_consecutive_format_errors <= self.n_consecutive_format_errors:
103                      self.add_messages(
104                          *e.m

`step()` 只是“一次迭代”：
```python
def step(self):
    return self.execute_actions(self.query())
```
记住这条边界：
```text
run  = 负责重复 step，直到退出
step = query + execute_actions
```
面试官问“Agent loop 在哪”，答 `run()`；问“一步做什么”，答 `query()` + `execute_actions()`。

## 用真实源码跑一次两步 Agent

仓库自带确定性测试模型 `DeterministicToolcallModel`——不联网、不花钱，按顺序吐固定回复。
下面让它第一步执行 `pwd`，第二步提交。重点看六条消息的 role。

In [24]:
from minisweagent.agents.default import DefaultAgent
from minisweagent.environments.local import LocalEnvironment
from minisweagent.models.test_models import DeterministicToolcallModel, make_toolcall_output

first = {"command": "pwd", "tool_call_id": "call_1"}
submit = {"command": "echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT\necho done", "tool_call_id": "call_2"}
outputs = [
    make_toolcall_output("先查看工作目录", [], [first]),
    make_toolcall_output("任务完成，提交", [], [submit]),
]
agent = DefaultAgent(
    DeterministicToolcallModel(outputs=outputs),
    LocalEnvironment(cwd=str(REPO)),
    system_template="你是一个可以使用 Bash 的编码助手。",
    instance_template="任务：{{task}}",
    cost_limit=5,
)
result = agent.run("查看工作目录，然后结束")
print("result =", result)
print("roles  =", [m.get("role") for m in agent.messages])
print(agent.messages)

result = {'exit_status': 'Submitted', 'submission': 'done\n'}
roles  = ['system', 'user', 'assistant', 'tool', 'assistant', 'exit']
[{'role': 'system', 'content': '你是一个可以使用 Bash 的编码助手。'}, {'role': 'user', 'content': '任务：查看工作目录，然后结束'}, {'role': 'assistant', 'content': '先查看工作目录', 'tool_calls': [], 'extra': {'actions': [{'command': 'pwd', 'tool_call_id': 'call_1'}], 'cost': 1.0, 'timestamp': 1784479170.941678}}, {'content': '<returncode>0</returncode>\n<output>\n/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent\n</output>', 'extra': {'raw_output': '/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent\n', 'returncode': 0, 'timestamp': 1784479170.957301, 'exception_info': ''}, 'tool_call_id': 'call_1', 'role': 'tool'}, {'role': 'assistant', 'content': '任务完成，提交', 'tool_calls': [], 'extra': {'actions': [{'command': 'echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT\necho done', 'tool_call_id': 'call_2'}], 'cost': 1.0, 'timestamp'

## 闭卷检查

不看源码，口头回答：
1. `run()` 和 `step()` 的职责差异？
2. 真正执行命令的对象是谁？
3. “模型发出完成信号”和“主循环退出”之间还差哪两步？

**完成标准**：你能画出 `run -> step -> (query -> model.query) + (execute_actions -> env.execute)` 这条链，且不把 `step()` 说成循环。